# Walkthrough: from survey export to dashboard

This notebook explains what the tool produces and how to check it. It reads **only the aggregate files in `outputs/`** (`findings.json`, `themes.json`). It never opens the survey export, so it shows no individual answers or comment text.

To refresh the files first, run `make run DATA=data/raw/<export>.xlsx` from the project folder (see `agents/tool/RUN.md`).

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "outputs" / "findings.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
findings = json.loads((ROOT / "outputs/findings.json").read_text(encoding="utf-8"))
themes = json.loads((ROOT / "outputs/themes.json").read_text(encoding="utf-8"))
T = findings["small_n_threshold"]

src = findings["source"]
print(f"Survey export: {src['n_rows']:,} rows, sheet '{src['sheet']}', fingerprint {src['file_sha256'][:12]}")
print(f"{len(findings['findings'])} published numbers; {len(themes['counts'])} theme counts; small-group threshold n < {T}")

## 1. Headline numbers

Every number has a stable **ID**. The memo, slides, video and dashboard all cite these IDs, and `scripts/check_numbers.py` fails if a draft shows a number that doesn't match. `n` is how many people the number is based on; the 95% range is where the true value most likely sits.

In [ ]:
F = pd.DataFrame(findings["findings"])
F["dimension"] = F["group"].map(lambda g: g["dimension"])
F["level"] = F["group"].map(lambda g: g["level"])
overall = F[F["dimension"] == "overall"][["id", "label", "value", "unit", "n", "ci_low", "ci_high", "caveat"]]
pd.set_option("display.max_colwidth", 70)
overall.round(2)

## 2. Small groups are never shown

A value based on fewer than 10 people is suppressed (value empty, n still reported) or pooled into "Other". The dashboard also hides any such value on its own, as a backstop. This cell checks the rule directly.

In [ ]:
leaks = F[(F["n"] < T) & F["value"].notna()]
print(f"Values from groups under {T} people that are not suppressed: {len(leaks)}")
print(f"Suppressed (value withheld, n shown): {int(F['suppressed'].sum())}")
print(f"Flagged small (n < {T}): {int(F['small_n'].sum())}")
assert leaks.empty, leaks[["id", "n"]]

## 3. Compare groups

After-training agreement by county, role and survey language. Read the gaps with the n and the 95% range: a group of 12 people has a much wider range than a group of 900.

In [ ]:
post = F[F["id"].str.match(r"^post\.[a-z0-9_]+\.agree_pct\.") & (F["dimension"] != "overall")].copy()
post["item"] = post["id"].str.split(".").str[1]
post["shown"] = post.apply(lambda r: f"{r.value:.1f}% (n={r.n:,}, {r.ci_low:.0f}-{r.ci_high:.0f})" if pd.notna(r.value) else f"not shown (n={r.n})", axis=1)
post.pivot_table(index=["dimension", "level"], columns="item", values="shown", aggfunc="first")

## 4. What people wrote

Comments were coded into themes by an offline, rule-based classifier (`agents/text/themes.py`; definitions in `agents/text/codebook.md`). A comment can carry several themes. The denominator is people who wrote something in that box. Catch-all buckets (filler, unmatched, questions in the "improve" box) are listed separately from topics.

In [ ]:
CATCH_ALL = {"no_substantive", "other_unclear", "audience_question"}
labels = {(t["field"], t["theme_id"]): t["label"] for t in themes["taxonomy"]}
C = pd.DataFrame([c for c in themes["counts"] if c["group"]["dimension"] == "overall"])
C["theme"] = [labels[(f, t)] for f, t in zip(C["field"], C["theme_id"])]
C["catch_all"] = C["theme_id"].isin(CATCH_ALL)
for field, g in C.groupby("field"):
    print(f"\n{field}: n = {g['n'].iloc[0]:,} comments")
    print(g.sort_values(["catch_all", "pct"], ascending=[True, False])[["theme", "n_mentions", "pct", "catch_all"]].to_string(index=False))

## 5. How the comment coder was checked

In [ ]:
v = themes["validation"]
print(f"Hand-checked sample: {v['sample_size']} comments; agreement {v['agreement_pct']}%; Cohen's kappa {v.get('cohen_kappa')}")
print(v["procedure"])

## 6. Rebuild the dashboard

`agents/tool/build.py` turns these two files into one self-contained page. Here it writes to a temporary file so this notebook changes nothing in `outputs/`. Open `outputs/dashboard.html` to see the committed version.

In [ ]:
import subprocess, sys, tempfile
out = Path(tempfile.mkdtemp()) / "dashboard.html"
r = subprocess.run([sys.executable, str(ROOT / "agents/tool/build.py"), "--strict", "--out", str(out)], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(f"{out.stat().st_size / 1024:.0f} KB, self-contained (no internet needed to view)")